In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
#movie_df = spark.read.parquet(f"{silver_folder_path}/movies").filter("year_release_date >= 2015")
movie_df = spark.read.table("movie_silver.movies").filter(f"file_date >= '{v_file_date}'")

In [0]:
#genre_df = spark.read.parquet(f"{silver_folder_path}/genre")
genre_df = spark.read.table("movie_silver.genre")

In [0]:
#movie_genre_df = spark.read.parquet(f"{silver_folder_path}/movies_genre")
movie_genre_df = spark.read.table("movie_silver.movies_genre").filter(f"file_date >= '{v_file_date}'")

In [0]:
movie_final_df = movie_df \
            .join(movie_genre_df, movie_df.movie_id == movie_genre_df.movie_id, "inner") \
            .join(genre_df, movie_genre_df.genre_id == genre_df.genre_id, "inner") \
            .select (movie_df.year_release_date, movie_df.budget, movie_df.revenue, genre_df.genre_name)

In [0]:
movie_filter_df = movie_final_df.filter("year_release_date >= 2015")

In [0]:
from pyspark.sql.functions import sum
results_group_by_df = movie_filter_df \
            .groupBy("year_release_date", "genre_name") \
            .agg(
                sum("budget").alias("budget"),
                sum("revenue").alias("revenue")
            )

In [0]:
from pyspark.sql.functions import lit

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, dense_rank, asc

result_dense_rank_df = Window.partitionBy("year_release_date").orderBy(
                                                                        desc("budget"),
                                                                        desc("revenue")
)
final_df = results_group_by_df.withColumn("dense_rank", dense_rank().over(result_dense_rank_df)) \
                            .withColumn("created_date", lit(v_file_date))

### Escribir datos en el DataLake en formato Parquet

In [0]:
#overwrite_partition("movie_gold", "results_group_movie_genre", "created_date", v_file_date)

In [0]:
#final_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_group_movie_genre")
#final_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_group_movie_genre")

merge_condition = 'tgt.year_release_date = src.year_release_date AND tgt.genre_name = src.genre_name AND tgt.created_date = src.created_date'
merge_delta_lake (final_df, "movie_gold", "results_group_movie_genre", merge_condition, "created_date")

In [0]:
%sql
SELECT *
FROM movie_gold.results_group_movie_genre

In [0]:
#display (spark.read.parquet(f"{gold_folder_path}/results_group_movie_genre"))